# Eight stores in a ring: testing the method on a known map

**Hayashi's Quantification Method IV** · a worked example from the StatNex teaching guide

**The story.** A chain has eight stores in a ring around a city centre, given as east-west and north-south positions in km. Before using quantification method IV on data with no map behind them, an analyst tries it on stores whose real layout is known.

**The question.** Does the method recover the layout of the stores around the centre?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/quantification_iv.csv` (8 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Store No.,East-west (km),North-south (km)
1.0,1.0,3.0
2.0,3.0,2.0
3.0,3.0,0.0
4.0,2.0,-3.0
5.0,-1.0,-3.0
6.0,-3.0,-2.0
7.0,-3.0,0.0
8.0,-2.0,3.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

pd.set_option("display.width", 120)

## The data

In [ ]:
# Text and blank cells count as missing, and a case missing any variable is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["East-west (km)", "North-south (km)"]
label_column = ["Store No."]
numbers = data[variables].apply(pd.to_numeric, errors="coerce")
complete = numbers.notna().all(axis=1)
x = numbers[complete].to_numpy(float)

# The app lays this table out like its regression methods. There is no response
# variable here, so its two rows are always 0.
summary = pd.DataFrame(
    {"n": [len(x), 0, len(data) - len(x), 0, len(data)]},
    index=["Valid cases", "Response only missing", "Explanatory only missing",
           "Both missing", "Total"],
)
summary["share"] = summary["n"] / len(data)
print(summary)

## Descriptive statistics and correlations

In [ ]:
# Variance and standard deviation divide by n - 1.
describe = pd.DataFrame(
    {"n": len(x), "mean": x.mean(axis=0), "variance": x.var(axis=0, ddof=1),
     "SD": x.std(axis=0, ddof=1), "min": x.min(axis=0), "max": x.max(axis=0)},
    index=variables,
)
correlation = np.corrcoef(x, rowvar=False)
print()
print(describe)
print()
print(pd.DataFrame(correlation, index=variables, columns=variables).round(4))

## The objects to place

In [ ]:
# The cases are placed, each described by its values on the variables.
objects = x
labels = (data.loc[complete, label_column[0]].astype(str).tolist() if label_column
          else [str(i + 1) for i in range(len(x))])
n = len(objects)

## Similarity matrix

In [ ]:
# The method needs a similarity for every pair -- large for two objects that belong
# together -- and this table gives measurements instead. So the distance between
# every pair is measured first (SciPy's pdist(), laid out as a table by
# squareform()), and the similarity is minus that distance: 0 on the diagonal, and
# the more negative, the further apart.
# Euclidean: the straight-line distance between two objects.
distances = squareform(pdist(objects, "euclidean"))
similarity = -distances
print()
print(pd.DataFrame(similarity, index=labels, columns=labels).round(4))

## H matrix

In [ ]:
# Hayashi's method gives each object a score x so that similar objects get close
# scores: it maximises -sum over pairs of (e_ij + e_ji)(x_i - x_j)^2 with the scores
# centred and their squares summing to 1. That sum is x'Hx for this H: off the
# diagonal, e_ij + e_ji; on it, minus the rest of the row, so each row sums to 0.
# With negated distances as the similarities, every pair carries a negative weight,
# so the method pushes the objects apart, the farthest pairs hardest. Read the
# diagonal: it is twice an object's total distance to all the others, and the
# objects with the largest diagonal end up nearest the edge of the map.
h = similarity + similarity.T
np.fill_diagonal(h, 0.0)
np.fill_diagonal(h, -h.sum(axis=1))
print()
print(pd.DataFrame(h, index=labels, columns=labels).round(4))

## Eigenvalues and eigenvectors

In [ ]:
# Each eigenvalue is the value of the sum above for its dimension's scores; the
# first dimension has the largest. There is no "share explained" here, unlike in
# MDS: compare the eigenvalues only to see whether the first two stand out.
# Because every row sums to 0, the constant vector is always an eigenvector, with
# eigenvalue 0: "put everything in one place", which says nothing. So the search is
# made among vectors orthogonal to it: the columns of `basis` span exactly those,
# H is solved inside that space, and the result is mapped back. Largest first, each
# turned to make its first element positive (a map and its mirror image are the
# same).
basis = np.linalg.qr(np.column_stack([np.ones(n), np.eye(n)[:, 1:]]))[0][:, 1:]
reduced = basis.T @ h @ basis
eigenvalues, inner = np.linalg.eigh((reduced + reduced.T) / 2)
order = np.argsort(-eigenvalues)
eigenvalues = eigenvalues[order]
vectors = basis @ inner[:, order]
vectors = vectors * np.where(vectors[0] < 0, -1.0, 1.0)
dims = [f"Dim {j + 1}" for j in range(n - 1)]
print()
print(pd.Series(eigenvalues, index=dims))
print()
print(pd.DataFrame(vectors, index=labels, columns=dims).round(4))

## The map

In [ ]:
# The first eigenvectors, as they are: unit-length scores with no units, so the map
# keeps which objects are near which, not the distances themselves.
dimensions = min(2, n - 1)
projection = vectors[:, :dimensions]
print()
print(pd.DataFrame(projection, index=labels, columns=dims[:dimensions]).round(4))

## What the result says

Quantification method IV places items on a map from their similarities alone. Here the similarity is the negative of the Euclidean (straight-line) distance between two stores, so near stores count as alike. The real layout is known, so the result can be checked.

The projection keeps the order around the ring: stores 1 to 8 follow each other on the map as they do on the ground, and opposite stores get opposite positions, such as store 2 at 0.53 and store 6 at -0.53 on dimension 1. But the positions are not kilometres, and the ring is turned. Stores 4 and 8 are also stretched outwards, to -0.62 and 0.62 on dimension 2. The first two eigenvalues, 93.7 and 89.6, are not far above the third, 76.2, so two dimensions do not hold everything.

So the method recovers the layout only partly. Use such a map for rough neighbourhoods, never for distances or directions.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.